# 1. Подготовка и нормализация данных

Исследование сравнивает два онлайн-каталога на 13 марта 2026 года. Здесь проверяем
замороженные таблицы, которые использовались для сопоставления, и разбираем этапы подготовки.

Рабочие эксперименты сохранены в `archive/`. Исходные цены и принятые автором метки не исправляются автоматически.
Подробности происхождения данных и ограничений — в `docs/DATA.md`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Работает при запуске Jupyter из корня проекта и из папки notebooks.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'processed').is_dir():
    raise FileNotFoundError('Открой Jupyter из корня проекта.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Два каталога

Названия могут отличаться порядком слов, написанием бренда, обозначениями объёма и упаковки.
Простое сравнение средних цен каталогов смешало бы различия в ценах с различиями ассортимента.

In [2]:
sk = pd.read_csv(ROOT / 'data/processed/sklavenitis_products.csv', keep_default_na=False)
am = pd.read_csv(ROOT / 'data/processed/alphamega_products.csv', keep_default_na=False)
pd.DataFrame({'магазин': ['Sklavenitis', 'AlphaMega'], 'товаров': [len(sk), len(am)]})

,магазин,товаров
0,Sklavenitis,3342
1,AlphaMega,8672


## Что извлекалось из названия

1. Нижний регистр, единые пробелы и удаление служебных фрагментов.
2. Вес, объём и количество единиц; приведение кг к граммам и литров к миллилитрам.
3. Дополнительные характеристики (`grade`) и текстовая основа названия (`product_core`).
4. Бренд и тип товара по словарям из 500 вручную размеченных строк.
5. Дополнительные ручные правила; для неоднозначных слов — ограничения по категории.

Алгоритм не гарантирует безошибочное извлечение каждого размера. Спорные совпадения проверялись вручную.

In [3]:
sk[['product_name', 'product_core', 'size_total_base', 'size_base_unit',
    'brand_pred_with_manual', 'product_type_pred_with_manual']].head(8)

,product_name,product_core,size_total_base,size_base_unit,brand_pred_with_manual,product_type_pred_with_manual
0,Spaghetti with Tomato Sauce SKLAVENITIS Produc...,spaghetti with tomato sauce sklavenitis produc...,480.0,g,sklavenitis,sauce
1,MACEDONIAN Halva with Almonds Vegan Gluten-Fre...,macedonian halva with almonds vegan gluten-free,400.0,g,macedonian,halva
2,"MAKEDONIKOS Halva with Cocoa – Vegan, Gluten F...",makedonikos halva with cocoa – vegan. gluten free,400.0,g,makedonikos,halva
3,OLYMPOS Halva with Pistachios – Gluten Free 40...,olympos halva with pistachios – gluten free,400.0,g,olympos,halva
4,MAKEDONIKOS Halva with Peanuts – Gluten Free 4...,makedonikos halva with peanuts – gluten free,400.0,g,makedonikos,peanuts
5,OLYMPOS Halva with Dark Chocolate & Almonds – ...,olympos halva with dark chocolate & almonds – ...,250.0,g,olympos,chocolate
6,MAKEDONIKOS Halva with Chocolate – Gluten Free...,makedonikos halva with chocolate – gluten free,40.0,g,makedonikos,chocolate
7,MAKEDONIKOS Halva Espresso – Gluten Free 40g 7...,makedonikos halva espresso – gluten free,40.0,g,makedonikos,halva


In [4]:
coverage = []
for store, frame in [('Sklavenitis', sk), ('AlphaMega', am)]:
    for field in ['brand_pred_with_manual', 'product_type_pred_with_manual']:
        coverage.append({'магазин': store, 'поле': field,
                         'покрытие, %': round(frame[field].ne('').mean() * 100, 2)})
pd.DataFrame(coverage)

,магазин,поле,"покрытие, %"
0,Sklavenitis,brand_pred_with_manual,85.37
1,Sklavenitis,product_type_pred_with_manual,92.43
2,AlphaMega,brand_pred_with_manual,48.58
3,AlphaMega,product_type_pred_with_manual,70.65


## Сохранённые ручные решения

Это входные данные исследования, а не ответы, которые нужно заново угадывать при запуске.
Правила брендов и типов лежат в CSV и доступны для проверки.

In [5]:
rules = pd.read_csv(ROOT / 'data/reference/product_type_manual_trigger_dictionary_current.csv').fillna('')
print('Правил типов товаров:', len(rules))
rules.head(8)

Правил типов товаров: 682


,product_type,trigger,category,category_keywords
0,smoked fish,smoked herring,,
1,dried fish,dried sea bream,lenten food,lenten | dried fish | fish | seafood
2,hummus,,,
3,pickled vegetables,pikla,,
4,pickled vegetables,gherkins,,
5,pickled vegetables,pickled celery,,
6,pickled vegetables,pickled beetroots,,
7,pickled vegetables,pickled mixed vegetables,,


## Необязательный запуск с исходных CSV

Публичная версия запускается с сохранённых обработанных таблиц. Если исходные CSV доступны локально,
положи их в `data/raw/` и установи флаг ниже в `True`. Результат появится в `data/interim/preparation/`,
а проверка различий — в `reports/preparation_validation.json`. Проверенные снимки не перезаписываются.
Исторические одноразовые правки и ручную разметку нельзя полностью повторить одной командой.

In [6]:
RUN_RAW_PREPARATION = False
if RUN_RAW_PREPARATION:
    from scripts.prepare_data import main as prepare_data
    preparation_check = prepare_data()
else:
    print('Используются сохранённые обработанные таблицы. Необязательный raw-этап пропущен.')

Используются сохранённые обработанные таблицы. Необязательный raw-этап пропущен.
